# Introduction

This experiment also addresses the *Airbus Ship Detection* task using a different architectural approach.

The Airbus dataset consists of high-resolution satellite images annotated with Run-Length Encoded (RLE) masks for each individual ship. Although the core task can be solved using semantic segmentation (predicting a single binary mask for ship vs. background) and U-Net performs strongly in this setting, as we have tested it before, the dataset’s instance-level nature raises the question of whether **object-wise detection models** can outperform classical segmentation architectures—especially when multiple ships appear in the same image.

To investigate this, we implemented a full **Mask R-CNN pipeline** using the PyTorch `torchvision` detection framework. Mask R-CNN extends Faster R-CNN by adding a per-object segmentation branch, making it able to:

1. **Detect each ship separately** via bounding box regression and object classification, and
2. **Predict a high-quality segmentation mask for each instance**.

These properties make Mask R-CNN a suitable architecture for datasets that explicitly encode multiple object instances per image.

The codebase developed for the Mask R-CNN experiment includes:

* **Model Definition:**
  A customized `maskrcnn_resnet50_fpn` model with classification and mask prediction heads modified to support binary (ship/background) output.

* **Dataset Handling:**
  A dedicated `ShipDataset` class that loads images, decodes RLE masks into binary arrays, computes bounding boxes, and generates PyTorch-compatible training targets.

* **Training Loop:**
  A complete training pipeline that computes all Mask R-CNN loss components—RPN losses, ROI classification and bounding box regression.

* **Evaluation:**
  Model predictions are evaluated using IoU-based matching and the **F2 score**, the same metric used in the original Airbus Kaggle challenge.

By implementing both U-Net and Mask R-CNN, we were able to compare **semantic segmentation vs. instance segmentation** approaches for satellite ship detection. While U-Net serves as a strong baseline, Mask R-CNN allows us to explore whether incorporating explicit object detection and per-instance reasoning leads to improved performance on this dataset. Unfortunately, this model below did not make it to be the primary submission solution, because it performed with a 0.08 F2 Score, scoring immensely lower than expected and than the 0.7 F-Score of our U-Net model. Due to the complexity of the model we found it harder to "debug" and experiment it and iterate over it, therefore we did not choose it as the primary branch of our development for the homework. But may it serve as an exciting example on exploring possible opportunities.


## One more thing

You may find our last and best checkpoint we have achieved with this model through this link: https://drive.google.com/file/d/1qsMGu4nLSngE0QfCCdAwkeOyQ4_tCsy0/view?usp=sharing

In [ ]:
import torchvision
from torchvision.models.detection.mask_rcnn import MaskRCNN_ResNet50_FPN_Weights, MaskRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.tensorboard import SummaryWriter

The purpose of the `get_model()` function is to create a Mask R-CNN model that is specifically adapted for our ship detection task. Instead of building the architecture from scratch, we start from a powerful pretrained model—`maskrcnn_resnet50_fpn`—which already learned general object features from the COCO dataset. This gives us a strong foundation and allows the model to converge much faster when training on the Airbus images.

However, the original COCO model predicts 80 object categories, while our project only needs two: **background** and **ship**. To address this mismatch, we replace the default classification and mask heads with new ones that output exactly the number of classes we need. These replacements ensure that the model focuses on the single object type relevant to our dataset, while still benefiting from all the pretrained feature extraction layers.

In short, this function takes a general-purpose COCO-trained Mask R-CNN and reshapes it into a specialized ship detector by keeping the powerful backbone and swapping only the model components that depend on the number of classes.


In [ ]:
def get_model(num_classes: int = 2) -> MaskRCNN:
    """ 
    Create a Mask R-CNN model for instance segmentation.

    :param num_classes: Number of output classes including background (default 2 → background + ship).
    :returns: Mask R-CNN model with a ResNet-50 FPN backbone, customized for the specified number of classes.
    """
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(
        weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = \
        torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden = 256
    model.roi_heads.mask_predictor = \
        torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
            in_features_mask, hidden, num_classes
        )

    return model

In [39]:
import torch
import torch.optim as optim

Once the model architecture is defined, we prepare it for training by instantiating it and moving it onto the chosen device. We then configure the optimizer and learning rate schedule, which are critical decisions for training stability and convergence. For this experiment, we use **stochastic gradient descent (SGD)** with a learning rate of **0.005**, momentum of **0.9**, and weight decay of **0.0005**. These values follow the standard configuration recommended for training Mask R-CNN–style models, balancing fast learning with enough regularization to avoid overfitting.

We also attach a **StepLR scheduler** that reduces the learning rate every three epochs by a factor of 0.1. This type of schedule helps the model transition from rapid initial learning to more fine-grained refinement later in training. Although this setup works well in practice, other configurations could have been explored—for example, using the Adam optimizer for potentially smoother training, cosine annealing for a more gradual learning rate decay, or experimenting with smaller initial learning rates to improve stability on small datasets.

Finally, a TensorBoard `SummaryWriter` is initialized to record losses throughout training. This allows us to visualize and compare training dynamics across different hyperparameter choices, making it easier to understand how each configuration affects model performance.

In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

model = get_model()
model.to(device)

optimizer = optim.SGD(
    model.parameters(),
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

writer = SummaryWriter(log_dir="runs/maskrcnn_train")


cpu


In [41]:

import cv2
import numpy as np

from utils import rle_to_mask
from torch.utils.data import Dataset
from typing import Dict, List, Optional, Any, Tuple

The `ShipDataset` class serves as the bridge between the Airbus dataset and the Mask R-CNN model. Since the original annotations are stored as run-length encoded (RLE) strings, the dataset must decode these masks on-the-fly and convert them into the specific format required by PyTorch’s detection API. Each image can contain multiple ships, so for every sample we must return not only the image tensor but also a structured target dictionary that includes all bounding boxes, instance masks, labels, and metadata.

Inside `__getitem__`, the dataset reads the image, normalizes it, and then loops through all RLE strings associated with that image. Each RLE mask is decoded into a binary mask, and from that mask we compute the tightest possible bounding box around the object. Only masks with non-zero area are kept, ensuring the model does not train on empty or corrupted annotations—a common issue in the Airbus dataset.

Finally, all masks and boxes are stacked and wrapped into tensors, and a label of `1` is assigned to every ship instance (since our dataset only has one foreground class). The resulting `target` dictionary matches the exact structure expected by Mask R-CNN. This design ensures that the model receives clean, consistent training samples while keeping dataset loading efficient and memory-friendly.


In [42]:
class ShipDataset(Dataset):
    """
    PyTorch Dataset for instance segmentation of ships.
    Loads images lazily and decodes RLE masks on-demand.
    Returns data in the format expected by Mask R-CNN.
    """

    def __init__(
        self,
        rle_dict: Dict[str, List[str]],
        image_root: str,
        transforms: Optional[Any] = None
    ):
        """
        Initialize the ShipDataset.

        :param rle_dict: Dictionary mapping image filenames to lists of RLE masks.
        :param image_root: Path to the folder containing images.
        :param transforms: Optional torchvision transforms to apply to the image.
        :returns: None
        """
        self.rle_dict = rle_dict
        self.image_root = image_root
        self.transforms = transforms
        self.img_ids = list(rle_dict.keys())

    def __len__(self) -> int:
        """
        Return the number of images in the dataset.

        :returns: Number of images as an integer.
        """
        return len(self.img_ids)

    def __getitem__(self, idx: int) -> Optional[Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """
        Load one image and all its instance masks in Mask R-CNN format.

        :param idx: Index of the image to load.
        :returns: Tuple of (image tensor, target dictionary) or None if no valid masks exist.
        """
        img_id = self.img_ids[idx]
        rles = self.rle_dict[img_id]

        img_path = f"{self.image_root}/{img_id}"
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(img_path)

        img = img[:, :, ::-1]
        img = img.astype(np.float32) / 255.0
        img_tensor = torch.from_numpy(img).permute(2, 0, 1)

        H, W = img_tensor.shape[1:]

        masks = []
        boxes = []

        for rle in rles:
            mask = rle_to_mask(rle, H, W).astype(np.uint8)
            
            if mask.sum() == 0:
                continue
            
            ys, xs = np.where(mask == 1)
            if len(xs) == 0:
                continue
            
            masks.append(mask)

            x1, y1 = xs.min(), ys.min()
            x2, y2 = xs.max(), ys.max()
            boxes.append([x1, y1, x2, y2])
        
        if len(boxes) == 0:
            return None

        masks = torch.as_tensor(np.stack(masks), dtype=torch.uint8)
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }

        return img_tensor, target

In [43]:
from tqdm import tqdm
from torch.utils.data import DataLoader

The `train_one_epoch()` function encapsulates the core loop that trains the model for a single epoch. Its job is simple: iterate over the dataset, run the model on each batch, compute the loss, and update the model’s weights. Before the loop begins, the model is placed in training mode, which activates certain layers like dropout or batch normalization in their learning configuration. A progress bar is also initialized so we can monitor how the loss evolves throughout the epoch.

For each batch, images and their corresponding target dictionaries are moved onto the correct device. Mask R-CNN expects lists of tensors rather than a single stacked batch, so both images and targets are kept in list form. When the model is called with these inputs, it returns a dictionary containing all its loss components—classification, bounding box regression, RPN losses, and mask loss. These individual losses are summed into a single scalar, which becomes the objective for backpropagation.

Finally, the optimizer updates the model parameters based on the computed gradients, and the running loss is accumulated for reporting. By returning the average loss across all batches, this function provides a clean numerical summary of training progress for each epoch, making it easy to track learning curves and adjust hyperparameters if necessary.

In [44]:
def train_one_epoch(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    data_loader: DataLoader,
    device: str,
    epoch: int
) -> float:
    """
    Train a Mask R-CNN model for one epoch.

    :param model: Mask R-CNN model.
    :param optimizer: PyTorch optimizer (SGD, Adam, etc.).
    :param data_loader: DataLoader yielding (images, targets).
    :param device: Device to run the training on ('cuda' or 'cpu').
    :param epoch: Current epoch number.
    :returns: Average loss over the epoch as a float.
    """

    model.train()
    total_loss: float = 0.0

    pbar = tqdm(data_loader, desc=f"Epoch {epoch}")

    for batch in pbar:
        
        if batch is None:
            continue

        images, targets = batch
        
        images: list[torch.Tensor] = [img.to(device) for img in images]

        targets: list[dict[str, torch.Tensor]] = [
            {key: val.to(device) for key, val in t.items()}
            for t in targets
        ]

        loss_dict: dict[str, torch.Tensor] = model(images, targets)

        losses: torch.Tensor = sum(loss_dict.values(), torch.tensor(0.0, device=device))
        
        total_loss += losses.item()

        optimizer.zero_grad()   
        losses.backward()
        optimizer.step()

        pbar.set_postfix(loss=float(losses.item()))

    return total_loss / len(data_loader)

In [45]:
import os

from torch.nn import Module

The `train()` function orchestrates the entire training process by repeatedly calling the single-epoch training routine and handling logging, scheduling, and checkpointing. At the start, it ensures that a directory exists for storing model checkpoints. Then, for each epoch, it delegates the core work to `train_one_epoch()`, which returns the average training loss—our primary indicator of learning progress.

After each epoch, the learning rate scheduler is stepped to gradually reduce the learning rate, which helps the model refine its predictions as training stabilizes. The function also logs the epoch loss to TensorBoard, allowing us to visually follow the training curve and quickly spot issues like stagnation or divergence. Each completed epoch results in a saved checkpoint, making it possible to resume training later or compare the performance of different model snapshots.

Once all epochs are finished, the TensorBoard writer is closed, and the training loop concludes. This function ties together the core components—training, scheduling, logging, and saving—into a single coherent pipeline that manages the model’s learning journey from start to finish.


In [46]:
def train(
    model: Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int
) -> None:
    """
    Train a Mask R-CNN model for multiple epochs, log losses, and save checkpoints.

    :param model: Mask R-CNN model to train.
    :param train_loader: DataLoader for training data.
    :param val_loader: DataLoader for validation data (unused in current code).
    :param epochs: Number of epochs to train.
    :returns: None
    """
    os.makedirs("checkpoints", exist_ok=True)
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)

        lr_scheduler.step()

        writer.add_scalar("Loss/train", train_loss, epoch)

        print(f"Epoch {epoch}/{epochs} - Loss: {train_loss:.4f}")

        torch.save(model.state_dict(), f"checkpoints/maskrcnn_epoch_{epoch}.pth")

    writer.close()


In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split

Before the model can be trained, we need to transform the Airbus annotation file into a structure that our dataset class can understand. The CSV file contains one row per ship mask, so the first step is to load it and group all RLE-encoded masks by their corresponding `ImageId`. This produces a dictionary where each image maps to a list of its ship masks—exactly the form required by the `ShipDataset` class.

Next, we construct a train/validation/test split by dividing the image IDs into subsets. A portion of the data is set aside for final testing, and the rest is evenly split between training and validation. This ensures that the model is trained on one subset of images and evaluated on completely unseen ones, making the performance estimates more reliable. The split also uses a fixed random seed for reproducibility.

Finally, we prepare the actual PyTorch dataset by passing the training portion of the RLE dictionary to our `ShipDataset` class. This dataset is now ready to feed images and instance annotations to the Mask R-CNN model during training.


In [48]:
df = pd.read_csv("../../data/segmentations.csv")

rle_dict = (
    df.groupby("ImageId")["EncodedPixels"]
      .apply(list)
      .to_dict()
)

im_ids = list(rle_dict.keys())

im_ids[:3]

test_im_ids, temp_im_ids = train_test_split(im_ids, test_size=0.4, random_state=42)
val_im_ids, train_im_ids = train_test_split(temp_im_ids, test_size=0.5, random_state=42)

train_rle_dict = {im_id: rle_dict[im_id] for im_id in train_im_ids}


shipDataset = ShipDataset(rle_dict=train_rle_dict,
                          image_root="../../data/images",
                          transforms=None)

In [49]:
from numpy.typing import NDArray

Mask R-CNN expects each batch to be a list of images and a matching list of target dictionaries, but our dataset occasionally returns `None` for images that contain no valid masks. The default PyTorch collation logic cannot handle these missing entries, so we define a custom `collate_fn` to clean up the batch before it reaches the model.

This function filters out all `None` samples and then rearranges the remaining elements into two tuples—one containing all images and the other containing all target dictionaries. This layout matches exactly what the Mask R-CNN forward pass expects. By handling incomplete samples gracefully, the custom collate function ensures that the DataLoader operates smoothly and the training loop remains robust even when some images have unusable annotations.


In [50]:
def collate_fn(
    batch: List[Tuple[NDArray, dict[str, NDArray]]]
) -> Optional[Tuple[Tuple[NDArray, ...], Tuple[dict[str, NDArray], ...]]]:
    """
    Custom collate function for PyTorch DataLoader that filters out None entries.

    :param batch: List of (image, target) tuples where image is a NumPy array and target is a dictionary of NumPy arrays.
    :returns: Tuple of zipped images and targets, or None if the batch is empty.
    """
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return tuple(zip(*batch))

Let's train the model!

In [51]:
train(
    model=model,
    train_loader=DataLoader(shipDataset, batch_size=4, shuffle=True, collate_fn=collate_fn),
    val_loader=None,
    epochs=5
)

Epoch 1:   0%|          | 3/9628 [00:11<9:56:45,  3.72s/it, loss=3.38] 


KeyboardInterrupt: 

In [ ]:
from typing import Dict, List
from utils import compute_iou_matrix, average_f_score_of_image, rles_to_masks

In [ ]:
@torch.inference_mode()
def evaluate_model(
    model: torch.nn.Module,
    rle_dict: Dict[str, List[str]],
    image_root: str,
    device: str = "cuda"
) -> Tuple[float, List[float]]:
    """
    Evaluate Faster-RCNN segmentation performance using F2 score.

    :param model: Trained Mask/Faster R-CNN model.
    :param rle_dict: Dictionary mapping image filenames to lists of ground truth RLE masks.
    :param image_root: Path to the folder containing images.
    :param device: Device to run inference on ('cuda' or 'cpu').
    :returns: Tuple of (average F2 score over all images, list of F2 scores per image).
    """
    model.eval()
    f2_scores = []

    for img_id, gt_rles in tqdm(rle_dict.items(), desc="Evaluating"):
        img_path = f"{image_root}/{img_id}"
        img_d = cv2.imread(img_path)
        
        if img_d is None:
            raise FileNotFoundError(img_path)
        
        img = img_d[:, :, ::-1]
        img_tensor = torch.from_numpy(img.astype(np.float32) / 255.).permute(2,0,1).to(device)

        pred = model([img_tensor])[0]  
        pred_masks = pred["masks"].squeeze(1).cpu().numpy() > 0.5

        H, W = img_tensor.shape[1:]
        gt_masks = rles_to_masks(gt_rles, H, W)

        iou_mat = compute_iou_matrix(gt_masks, pred_masks)

        f2 = average_f_score_of_image(iou_mat)
        f2_scores.append(f2)

    return np.mean(f2_scores), f2_scores

In [ ]:
_mean_f2, f2_scores = evaluate_model(
    model=model,
    rle_dict={im_id: rle_dict[im_id] for im_id in val_im_ids},
    image_root="./data/images",
    device="cuda"
)

print(f"Mean F2 Score on Validation Set: {_mean_f2:.4f}")

## Conclusion

After evaluation on unseen data, we were sorry to see that this model did not generalize well *at all* with an F2-Score of 0.08. It could have been that 5 epochs was not enough for it to learn, or that it learned completely irrelevant features. The next steps would have been to perhaps modify the images (variate contrast, for example), train the model for more epochs. However, after some research we have found that other Kaggle users were more successful with U-Net, for example, therefore for now we deemed this model a "lost cause", something worthwile to revisit later, but the potential gains from a U-Net model were larger than sticking to this architecture, as it might be flawed overall for this specific dataset.